# Local ReasonIF: base vs DPO-only vs SFT-to-DPO at 8192 tokens

This notebook compares three Qwen3-0.6B checkpoints on the same balanced 300-question ReasonIF set: the base model, the full DPO-only model, and the final mixed-SFT-to-DPO model.

Generation uses a low temperature (`0.1`), an 8192-new-token ceiling, fixed seeds, identical prompts, and official ReasonIF scoring. Outputs are written outside OneDrive and saved atomically so interrupted runs can resume.

## 1. Select the CUDA-enabled local kernel

In VS Code, select the local Python environment containing CUDA-enabled PyTorch. This notebook intentionally does not reinstall PyTorch.

In [ ]:
import platform
import sys
from pathlib import Path

import torch

print("Python:", sys.executable)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), (
    "CUDA is unavailable. In VS Code select the local kernel containing your "
    "CUDA-enabled PyTorch installation, then restart the notebook kernel."
)
gpu_name = torch.cuda.get_device_name(0)
gpu_bytes = torch.cuda.get_device_properties(0).total_memory
print("GPU:", gpu_name, f"({gpu_bytes / 2**30:.1f} GiB)")
assert "4080" in gpu_name, "This configuration was sized for an RTX 4080."
assert gpu_bytes >= 14 * 2**30, "Use the 16 GB RTX 4080 rather than a lower-memory GPU."

## 2. Install evaluation dependencies

Restart the kernel only if VS Code explicitly requests it after installation.

In [ ]:
%pip install -q -U "transformers>=4.51,<4.57" "accelerate>=1.2" safetensors sentencepiece pandas matplotlib tqdm sympy pylatexenc fast-langdetect nltk immutabledict packaging

## 3. Editable settings

The notebook checks the Colab path, a Google Drive Desktop G: mount, and models/ inside this repository. On the local RTX 4080, make the final DPO model folder available through Drive Desktop or copy it into models/.

Start with NUM_QUESTIONS = 30. Change it to 300 for the full ReasonIF evaluation.

In [ ]:
import os
import re

PROJECT = Path.cwd().resolve()
REASONIF_ROOT = PROJECT / "reasonIF"

DPO_ONLY_FOLDER_NAME = "qwen3-0.6b-all-controls-5k-dpo-2epochs"
DPO_ONLY_CANDIDATES = [
    Path(r"G:\My Drive\CoT_Controllability") / DPO_ONLY_FOLDER_NAME,
    PROJECT / "models" / DPO_ONLY_FOLDER_NAME,
]
DPO_ONLY_ROOT = next(
    (path for path in DPO_ONLY_CANDIDATES if path.is_dir()),
    DPO_ONLY_CANDIDATES[-1],
)

SFT_DPO_FOLDER_NAME = "qwen3-0.6b-mixed-sft-then-controls-dpo-cutoff4096-1epoch"
SFT_DPO_CANDIDATES = [
    Path(r"G:\My Drive\CoT_Controllability") / SFT_DPO_FOLDER_NAME,
    PROJECT / "models" / SFT_DPO_FOLDER_NAME,
]
SFT_DPO_ROOT = next(
    (path for path in SFT_DPO_CANDIDATES if path.is_dir()),
    SFT_DPO_CANDIDATES[-1],
)

BASE_MODEL = "Qwen/Qwen3-0.6B"
NUM_QUESTIONS = 300
QUESTION_SEED = 42
GENERATION_SEED = 42
MAX_NEW_TOKENS = 8192
MAX_INPUT_TOKENS = 2048
BATCH_SIZE = 1  # Safest setting for an RTX 4080 with an 8192-token ceiling.
TEMPERATURE = 0.1
TOP_P = 0.95

assert REASONIF_ROOT.is_dir(), f"ReasonIF repository not found: {REASONIF_ROOT}"
assert (REASONIF_ROOT / "data/reasonIF_dataset.json").is_file()
assert 1 <= NUM_QUESTIONS <= 300
assert BATCH_SIZE >= 1

run_key = (
    f"n{NUM_QUESTIONS}_seed{QUESTION_SEED}_new{MAX_NEW_TOKENS}_"
    f"temp{TEMPERATURE:g}"
)
LOCAL_RESULTS_BASE = Path(os.environ.get(
    "COT_RESULTS_ROOT",
    Path.home() / "Downloads/CoT_Controllability_outputs",
)).resolve()
RESULTS_ROOT = LOCAL_RESULTS_BASE / "reasonif_three_model_8192_comparison" / run_key
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Results:", RESULTS_ROOT)


## 4. Validate the final model

The evaluator loads the final weights from the top level of the SFT→DPO output folder. Training checkpoints are not required.

In [ ]:
def has_model_weights(path):
    path = Path(path)
    return (
        (path / "config.json").is_file()
        and (any(path.glob("*.safetensors")) or any(path.glob("pytorch_model*.bin")))
    )

for label, root, candidates in [
    ("DPO only", DPO_ONLY_ROOT, DPO_ONLY_CANDIDATES),
    ("Final SFT -> DPO", SFT_DPO_ROOT, SFT_DPO_CANDIDATES),
]:
    if not has_model_weights(root):
        checked = "\n".join(f"  {candidate}" for candidate in candidates)
        raise FileNotFoundError(
            f"{label} model weights were not found. Checked:\n{checked}"
        )

MODEL_SPECS = {
    "Base Qwen3-0.6B": BASE_MODEL,
    "DPO only (2 epochs)": str(DPO_ONLY_ROOT.resolve()),
    "Final SFT -> DPO": str(SFT_DPO_ROOT.resolve()),
}
print("Models to evaluate:")
for label, path in MODEL_SPECS.items():
    print(f"  {label}: {path}")


## 5. Select one balanced, deterministic ReasonIF subset

When fewer than 300 questions are requested, the sampler spreads examples as evenly as possible across all six constraints. Every model receives the same rows in the same order.

In [ ]:
import json
import random
from collections import defaultdict

dataset = json.loads(
    (REASONIF_ROOT / "data/reasonIF_dataset.json").read_text(encoding="utf-8")
)
assert len(dataset) == 300

groups = defaultdict(list)
for index, row in enumerate(dataset):
    groups[row["constraint_name"][0]].append(index)
rng = random.Random(QUESTION_SEED)
for indices in groups.values():
    rng.shuffle(indices)

selected_indices = []
ordered_constraints = sorted(groups)
while len(selected_indices) < NUM_QUESTIONS:
    added = False
    for constraint in ordered_constraints:
        if groups[constraint] and len(selected_indices) < NUM_QUESTIONS:
            selected_indices.append(groups[constraint].pop())
            added = True
    if not added:
        break

selected_rows = [dict(dataset[index], dataset_index=index) for index in selected_indices]
selection_manifest = {
    "num_questions": NUM_QUESTIONS,
    "seed": QUESTION_SEED,
    "dataset_indices": selected_indices,
    "constraint_counts": {
        constraint: sum(row["constraint_name"][0] == constraint for row in selected_rows)
        for constraint in ordered_constraints
    },
}
(RESULTS_ROOT / "selection_manifest.json").write_text(
    json.dumps(selection_manifest, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(selection_manifest["constraint_counts"], indent=2))

## 6. Resumable local generation

Each response is appended immediately to its model's JSONL file. If generation is interrupted, rerun this cell and completed dataset indices will be skipped.

In [ ]:
import gc
import hashlib
import time

from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

def safe_label(label):
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")

def split_reasoning(raw_output):
    text = raw_output.strip()
    if "</think>" in text:
        reasoning, content = text.split("</think>", 1)
        reasoning = reasoning.removeprefix("<think>").strip()
        return reasoning, content.strip()
    if text.startswith("<think>"):
        return text[len("<think>"):].strip(), ""
    return text, ""

def read_completed(path, max_attempts=5):
    # OneDrive can briefly expose a partially synchronized JSONL record.
    # Read and parse a fresh snapshot, retrying only transient JSON errors.
    import time
    path = Path(path)
    if not path.exists():
        return {}
    for attempt in range(1, max_attempts + 1):
        completed = {}
        snapshot = path.read_bytes().decode("utf-8")
        try:
            # JSONL records are separated by LF only. str.splitlines() also
            # splits valid JSON strings containing U+2028/U+2029.
            for line_number, line in enumerate(snapshot.split("\n"), 1):
                if not line.strip():
                    continue
                row = json.loads(line)
                completed[row["dataset_index"]] = row
            return completed
        except json.JSONDecodeError as error:
            if attempt == max_attempts:
                raise ValueError(
                    f"Invalid JSONL after {max_attempts} fresh reads at "
                    f"{path}:{line_number}: {error}"
                ) from error
            time.sleep(0.5)

def generate_model(label, model_path):
    torch.manual_seed(GENERATION_SEED)
    torch.cuda.manual_seed_all(GENERATION_SEED)
    output_path = RESULTS_ROOT / f"{safe_label(label)}_raw.jsonl"
    completed = read_completed(output_path)
    pending = [row for row in selected_rows if row["dataset_index"] not in completed]
    print(f"\n{label}: {len(completed)}/{len(selected_rows)} already saved; {len(pending)} pending")
    if not pending:
        return output_path

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
        trust_remote_code=True,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    ).eval()

    for start in tqdm(range(0, len(pending), BATCH_SIZE), desc=label):
        batch = pending[start:start + BATCH_SIZE]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": row["prompt"]}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
            for row in batch
        ]
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to("cuda")
        generation_kwargs = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        if TEMPERATURE > 0:
            generation_kwargs.update(
                do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
            )
        else:
            generation_kwargs["do_sample"] = False

        started = time.time()
        with torch.inference_mode():
            generated = model.generate(**encoded, **generation_kwargs)
        prompt_width = encoded["input_ids"].shape[1]
        continuation_ids = generated[:, prompt_width:]
        texts = tokenizer.batch_decode(continuation_ids, skip_special_tokens=False)
        elapsed = time.time() - started

        for row, token_ids, raw in zip(batch, continuation_ids, texts):
            raw = raw.replace(tokenizer.pad_token or "", "").strip()
            reasoning, content = split_reasoning(raw)
            output_tokens = int((token_ids != tokenizer.pad_token_id).sum().item())
            record = {
                "dataset_index": row["dataset_index"],
                "model_label": label,
                "model_path": str(model_path),
                "source": row["source"],
                "question": row["question"],
                "answer": row["answer"],
                "constraint_name": row["constraint_name"],
                "constraint_args": row["constraint_args"],
                "prompt": row["prompt"],
                "raw_output": raw,
                "reasoning_content": reasoning,
                "content": content,
                "output_tokens": output_tokens,
                "truncated": output_tokens >= MAX_NEW_TOKENS,
                "batch_seconds": elapsed,
            }
            # Commit the whole JSONL update atomically. If generation is
            # interrupted, the previous valid file remains untouched.
            serialized = (json.dumps(record, ensure_ascii=False) + "\n").encode("utf-8")
            existing = output_path.read_bytes() if output_path.exists() else b""
            temp_path = output_path.with_suffix(output_path.suffix + ".tmp")
            with temp_path.open("wb") as sink:
                sink.write(existing)
                sink.write(serialized)
                sink.flush()
                os.fsync(sink.fileno())
            os.replace(temp_path, output_path)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return output_path

RAW_PATHS = {}
for model_label, model_path in MODEL_SPECS.items():
    RAW_PATHS[model_label] = generate_model(model_label, model_path)
print("\nGeneration files:")
for label, path in RAW_PATHS.items():
    print(label, "->", path)

## 7. Score with the official ReasonIF checkers

Instruction following is evaluated on the extracted reasoning trace, exactly as in ReasonIF. Answer accuracy uses the repository's source-specific final-answer extractor and requires the model to place its answer in `<answer>...</answer>`.

In [ ]:
if str(REASONIF_ROOT) not in sys.path:
    sys.path.insert(0, str(REASONIF_ROOT))

from src.eval_utils import evaluate_instruction_following, extract_final_answer
import src.instructions.instruction_checker as reasonif_instruction_checker
from fast_langdetect import detect as installed_language_detect

# ReasonIF originally called fast-langdetect's older
# `detect(text, low_memory=False) -> dict` API. Current releases use
# `detect(text, model="full") -> list[dict]`. Adapt only that interface while
# retaining the full language model used by the original checker.
def reasonif_compatible_detect(text, low_memory=False):
    try:
        result = installed_language_detect(text, low_memory=low_memory)
    except TypeError as error:
        if "low_memory" not in str(error):
            raise
        result = installed_language_detect(
            text, model="lite" if low_memory else "full"
        )
    if isinstance(result, list):
        if not result:
            raise ValueError("fast-langdetect returned no predictions")
        return result[0]
    return result

reasonif_instruction_checker.detect = reasonif_compatible_detect
print("ReasonIF language checker compatibility enabled.")

def canonical_answer(value, source):
    value = str(value).strip().replace(",", "")
    if source in {"arc", "gpqa"}:
        match = re.search(r"[ABCD]", value.upper())
        return match.group(0) if match else value.upper()
    try:
        number = float(value)
        return str(int(number)) if number.is_integer() else f"{number:.12g}"
    except ValueError:
        return value

scored_rows = []
for label, path in RAW_PATHS.items():
    records = list(read_completed(path).values())
    assert len(records) == len(selected_rows), f"{label} is incomplete"
    for record in records:
        reasoning = record["reasoning_content"]
        follows = all(evaluate_instruction_following(
            instruction_id_list=record["constraint_name"],
            parameters=record["constraint_args"],
            prompt=record["question"],
            response=reasoning,
        )) if reasoning.strip() else False
        predicted = extract_final_answer(record["content"], record["source"])
        answer_correct = (
            canonical_answer(predicted, record["source"])
            == canonical_answer(record["answer"], record["source"])
        )
        scored_rows.append({
            **record,
            "model": label,
            "constraint": record["constraint_name"][0],
            "predicted_answer": predicted,
            "instruction_following": follows,
            "answer_correct": answer_correct,
        })

SCORED_PATH = RESULTS_ROOT / "scored_responses.jsonl"
SCORED_PATH.write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in scored_rows),
    encoding="utf-8",
)
print("Scored responses:", SCORED_PATH)

## 8. Print overall and per-category results

These tables report instruction following and answer accuracy for the final SFT→DPO model.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

frame = pd.DataFrame(scored_rows)
# During a resumable run, report only models present in scored_rows.
available_models = set(frame["model"].dropna().unique())
model_order = [model for model in MODEL_SPECS if model in available_models]

overall = frame.groupby("model", sort=False).agg(
    instruction_following=("instruction_following", "mean"),
    answer_accuracy=("answer_correct", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    truncated=("truncated", "sum"),
    questions=("dataset_index", "count"),
).reindex(model_order)
overall_display = overall.copy()
for column in ["instruction_following", "answer_accuracy"]:
    overall_display[column] = overall_display[column].map(lambda value: f"{value:.1%}")
overall_display["mean_output_tokens"] = overall_display["mean_output_tokens"].round(0).astype("Int64")
display(overall_display)

category_long = frame.groupby(["constraint", "model"], sort=False).agg(
    instruction_following=("instruction_following", "mean"),
    answer_accuracy=("answer_correct", "mean"),
    questions=("dataset_index", "count"),
).reset_index()
category_if = category_long.pivot(index="constraint", columns="model", values="instruction_following").reindex(columns=model_order)
category_accuracy = category_long.pivot(index="constraint", columns="model", values="answer_accuracy").reindex(columns=model_order)

print("Instruction following by constraint")
display(category_if.map(lambda value: f"{value:.1%}"))
print("Answer accuracy by constraint")
display(category_accuracy.map(lambda value: f"{value:.1%}"))

overall.to_csv(RESULTS_ROOT / "overall_comparison.csv")
category_long.to_csv(RESULTS_ROOT / "category_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(max(11, len(model_order) * 2.2), 4.5))
overall["instruction_following"].plot.bar(ax=axes[0], ylim=(0, 1), title="ReasonIF instruction following")
overall["answer_accuracy"].plot.bar(ax=axes[1], ylim=(0, 1), title="ReasonIF answer accuracy")
for axis in axes:
    axis.set_xlabel("")
    axis.set_ylabel("score")
    axis.grid(axis="y", alpha=0.2)
    axis.tick_params(axis="x", rotation=30)
fig.tight_layout()
FIGURE_PATH = RESULTS_ROOT / "overall_comparison.png"
fig.savefig(FIGURE_PATH, dpi=170, bbox_inches="tight")
plt.show()
print("Saved tables and figure to:", RESULTS_ROOT)

## 9. Inspect individual failures

Change the filters to inspect incoherence, truncation, instruction failures, or wrong answers before choosing the DPO starting checkpoint.

In [ ]:
MODEL_TO_INSPECT = model_order[-2]  # Change to any label in model_order
ONLY_WRONG_ANSWERS = True
ONLY_IF_FAILURES = False
ONLY_TRUNCATED = False

inspection = frame[frame["model"] == MODEL_TO_INSPECT].copy()
if ONLY_WRONG_ANSWERS:
    inspection = inspection[~inspection["answer_correct"]]
if ONLY_IF_FAILURES:
    inspection = inspection[~inspection["instruction_following"]]
if ONLY_TRUNCATED:
    inspection = inspection[inspection["truncated"]]

print(f"Matching rows: {len(inspection)}")
for _, row in inspection.head(10).iterrows():
    print("\n" + "=" * 100)
    print("MODEL:", row["model"], "| CONSTRAINT:", row["constraint"], "| SOURCE:", row["source"])
    print("GOLD:", repr(row["answer"]), "| PREDICTED:", repr(row["predicted_answer"]))
    print("IF:", row["instruction_following"], "| CORRECT:", row["answer_correct"], "| TRUNCATED:", row["truncated"])
    print("QUESTION:\n", row["question"])
    print("REASONING:\n", row["reasoning_content"])
    print("FINAL CONTENT:\n", row["content"])